# Assignment 2.1 — CNN and Pre-Trained Models

**Purpose:** Build a CNN for handwritten-digit OCR using MNIST, train a CNN on CIFAR-10 with image augmentation, improve generalization, and compare two ImageNet-pretrained models using transfer learning.

**How to use this notebook**
1. Run the cells from top to bottom in order.
2. The first run may download MNIST, CIFAR-10, and ImageNet weights for VGG16 and MobileNetV2.
3. Training cells can take time, especially the pretrained VGG16 model.
4. Outputs have been cleared because the training/evaluation workflow was revised. Re-run the notebook to generate fresh curves and final test metrics.

**Important evaluation practice:** A validation subset is separated from the training data. The test set is held back for final evaluation only, rather than being repeatedly used to guide training.

### Cell 1 — Imports, random seed, and MNIST loading
This cell imports the main libraries, sets random seeds to make results more reproducible, and loads MNIST from TensorFlow/Keras. MNIST contains grayscale images of handwritten digits from 0 to 9. The printed shapes confirm the number of images and their dimensions.

In [ ]:
# Reproducibility and dataset loading
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load MNIST: 60,000 training and 10,000 test grayscale digit images.
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)

### Cell 2 — Visualize MNIST (Part 1.a)
This cell displays a 4 × 4 sample of handwritten digits. The grayscale colormap is appropriate because MNIST images contain one intensity channel. The label above each image is the correct target class. This visual check helps confirm that the dataset loaded correctly.

In [ ]:
# Visualize a sample of MNIST images
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle(
    "Part 1.a: Sample MNIST Handwritten Digits",
    fontsize=14,
    fontweight="bold",
)

for idx, ax in enumerate(axes.flat):
    ax.imshow(X_train[idx], cmap="gray")
    ax.set_title(f"Label: {y_train[idx]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Cell 3 — Normalize and reshape MNIST
Neural networks train more reliably when pixel values are scaled to a small numeric range. Dividing by 255 converts pixels from 0–255 to 0–1. The extra final dimension changes each image from 28 × 28 to 28 × 28 × 1, which is the channels-last input format expected by the CNN's Conv2D layers.

In [ ]:
# Prepare MNIST images for a 2D CNN
# Pixel values are converted from [0, 255] to [0, 1].
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

# Conv2D with channels_last expects (batch, height, width, channels).
X_train_last = np.expand_dims(X_train_norm, axis=-1)
X_test_last = np.expand_dims(X_test_norm, axis=-1)

print("Processed training shape:", X_train_last.shape)
print("Processed test shape:", X_test_last.shape)
print("Pixel value range:", X_train_last.min(), "to", X_train_last.max())

### Cell 4 — Build the MNIST CNN (Part 1.b)
The network has exactly four convolution layers: two with 32 filters and two with 64 filters. Max-pooling reduces the spatial dimensions. Flatten converts the feature maps into a vector, followed by the required 512-unit hidden dense layer and a 10-unit softmax output for digit classes. Sparse categorical cross-entropy is suitable because labels are integer class IDs.

In [ ]:
# Build and compile the Part I CNN
# Four convolution layers: 32, 32, 64, and 64 filters.
# The classifier has one 512-unit hidden dense layer and 10 output classes.
IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS = 28, 28, 1

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])

# Adam is used because Part I does not prescribe a specific optimizer.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

### Cell 5 — Train and test the MNIST CNN
The model is trained for 20 epochs with batch size 128, as requested. Ten percent of the training data is used for validation. The separate 10,000-image test set is evaluated only after training, providing a final estimate of performance on unseen data.

In [ ]:
# Train the MNIST CNN
# Keep 10% of the training set for validation. The separate test set is
# reserved for the final unbiased evaluation.
BATCH_SIZE = 128
EPOCHS = 20

history = model.fit(
    X_train_last,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    shuffle=True,
)

mnist_test_loss, mnist_test_accuracy = model.evaluate(
    X_test_last, y_test, verbose=0
)
print(f"MNIST test accuracy: {mnist_test_accuracy:.4f}")

### Cell 6 — MNIST learning curves and interpretation
The loss curves show prediction error; lower is better. Accuracy curves show the fraction of correctly classified images; higher is better. If training and validation curves improve together and remain close, the model is generalizing. If training improves while validation stalls or worsens, the model is likely overfitting. Use the actual plotted curves and test accuracy when writing the final observation.

In [ ]:
# Plot and interpret the MNIST learning curves
def plot_learning_curves(history, epochs, loss_title, accuracy_title):
    epoch_numbers = range(1, len(history.history["loss"]) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epoch_numbers, history.history["loss"], label="Training loss")
    plt.plot(epoch_numbers, history.history["val_loss"], label="Validation loss")
    plt.title(loss_title)
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epoch_numbers, history.history["accuracy"], label="Training accuracy")
    plt.plot(epoch_numbers, history.history["val_accuracy"], label="Validation accuracy")
    plt.title(accuracy_title)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_learning_curves(
    history,
    EPOCHS,
    "MNIST CNN: Training vs Validation Loss",
    "MNIST CNN: Training vs Validation Accuracy",
)

### Cell 7 — Load CIFAR-10
This cell loads CIFAR-10, which has 50,000 training and 10,000 test images. Unlike MNIST, each image is a 32 × 32 RGB color image with three channels. The labels are integer IDs for ten object categories.

In [ ]:
# Load the CIFAR-10 dataset
# CIFAR-10 contains 50,000 training and 10,000 test color images (32x32x3).
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = (
    tf.keras.datasets.cifar10.load_data()
)

print("Training images:", x_train_cifar.shape)
print("Training labels:", y_train_cifar.shape)
print("Test images:", x_test_cifar.shape)
print("Test labels:", y_test_cifar.shape)

### Cell 8 — Visualize the first nine CIFAR-10 images (Part 2.a)
The class-name list maps each integer label to a readable category. The 3 × 3 plot displays the first nine color images and their labels. This is a quick visual check of image content, color channels, and label mapping.

In [ ]:
# Visualize the first nine CIFAR-10 images and their class labels
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

fig, axes = plt.subplots(3, 3, figsize=(7, 7))
fig.suptitle("Part 2.a: First 9 CIFAR-10 Images", fontsize=14, fontweight="bold")

for idx, ax in enumerate(axes.flat):
    ax.imshow(x_train_cifar[idx])
    class_idx = int(y_train_cifar[idx, 0])
    ax.set_title(class_names[class_idx])
    ax.axis("off")

plt.tight_layout()
plt.show()

### Cell 9 — Normalize and augment CIFAR-10 (Part 2.b)
Pixel values are scaled to 0–1. The training generator applies the requested 20% shear, 20% zoom, and random horizontal flip, creating varied versions of training images. A separate, unaugmented validation subset is used to monitor generalization. The test set is not used for training or model selection.

In [ ]:
# Normalize CIFAR-10 and configure the required image augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

x_train_cifar_norm = x_train_cifar.astype("float32") / 255.0
x_test_cifar_norm = x_test_cifar.astype("float32") / 255.0

# Apply the requested transformations only to training images:
# shear by 20%, zoom by 20%, and random horizontal flipping.
# Split off 10% of the original training data for validation.
train_datagen = ImageDataGenerator(
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.1,
)

# Validation images are not augmented, so validation metrics are easier to interpret.
validation_datagen = ImageDataGenerator(validation_split=0.1)

train_generator = train_datagen.flow(
    x_train_cifar_norm,
    y_train_cifar,
    batch_size=128,
    subset="training",
    seed=SEED,
    shuffle=True,
)

validation_generator = validation_datagen.flow(
    x_train_cifar_norm,
    y_train_cifar,
    batch_size=128,
    subset="validation",
    seed=SEED,
    shuffle=False,
)

print("Normalized training shape:", x_train_cifar_norm.shape)
print("Normalized test shape:", x_test_cifar_norm.shape)
print("Training batches per epoch:", len(train_generator))
print("Validation batches:", len(validation_generator))

### Cell 10 — Build the baseline CIFAR-10 CNN (Part 2.b)
This architecture follows the assignment: four convolution layers with 32, 32, 64, and 64 filters, then dense hidden layers of 512 and 256 units, and a 10-class softmax output. SGD uses learning rate 0.005, decay 1 × 10⁻⁶, and momentum 0.9. In current Keras, InverseTimeDecay expresses the legacy inverse-time decay schedule.

In [ ]:
# Build the baseline CIFAR-10 CNN for Part 2.b
# Four convolution layers (32, 32, 64, 64) and two dense hidden layers
# (512 and 256 units), as required by the assignment.
model_cifar = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])

# Keras 3 equivalent of the legacy SGD decay parameter:
# lr(step) = 0.005 / (1 + 1e-6 * step), with momentum 0.9.
lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=0.005,
    decay_steps=1,
    decay_rate=1e-6,
    staircase=False,
)
sgd_optimizer = tf.keras.optimizers.SGD(
    learning_rate=lr_schedule,
    momentum=0.9,
)

model_cifar.compile(
    optimizer=sgd_optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model_cifar.summary()

### Cell 11 — Train the baseline CIFAR-10 CNN
The model learns from augmented batches of images. It runs for 20 epochs with batch size 128. Validation metrics are calculated on the held-out, unaugmented validation subset. The final test accuracy is calculated separately after fitting.

In [ ]:
# Train the baseline CIFAR-10 CNN using augmented images
BATCH_SIZE = 128
EPOCHS = 20

history = model_cifar.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
)

# Evaluate once on the held-out test set after training.
cifar_baseline_test_loss, cifar_baseline_test_accuracy = model_cifar.evaluate(
    x_test_cifar_norm, y_test_cifar, verbose=0
)
print(f"Baseline CNN test accuracy: {cifar_baseline_test_accuracy:.4f}")

### Cell 12 — Baseline CIFAR-10 learning curves
These plots compare training and validation loss/accuracy over the epochs. Look for a widening gap: high training accuracy with lower or stagnant validation accuracy, especially alongside rising validation loss, is evidence of overfitting. This observation motivates Part 2.c.

In [ ]:
# Plot Part 2.b learning curves
plot_learning_curves(
    history,
    EPOCHS,
    "CIFAR-10 Baseline CNN: Loss",
    "CIFAR-10 Baseline CNN: Accuracy",
)

### Cell 13 — Part 2.c: Improve generalization
This model adds Batch Normalization, Dropout, and L2 weight regularization while keeping the four-convolution-layer pattern. Batch Normalization stabilizes intermediate activations. Dropout randomly disables a fraction of units during training, and L2 discourages excessively large weights. Early Stopping restores the weights from the best validation-loss epoch. These are regularization techniques intended to reduce overfitting; compare the resulting curves and test score with the baseline rather than assuming improvement.

In [ ]:
# Part 2.c: Improve generalization with Batch Normalization and Dropout
# The baseline model can overfit when training accuracy keeps rising while
# validation accuracy plateaus and validation loss stops improving.
# Batch Normalization stabilizes activations; Dropout and L2 regularization
# reduce reliance on individual units and discourage overly large weights.

from tensorflow.keras import regularizers

model_cifar_v2 = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(
        512, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
    ),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(
        256, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
    ),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation="softmax"),
])

# Keep the same SGD configuration so the comparison focuses on the
# regularization / normalization changes rather than a different optimizer.
lr_schedule_v2 = tf.keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=0.005,
    decay_steps=1,
    decay_rate=1e-6,
    staircase=False,
)
sgd_optimizer_v2 = tf.keras.optimizers.SGD(
    learning_rate=lr_schedule_v2,
    momentum=0.9,
)

model_cifar_v2.compile(
    optimizer=sgd_optimizer_v2,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model_cifar_v2.summary()

# Stop if validation loss no longer improves; restore the best observed weights.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

history_v2 = model_cifar_v2.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=[early_stopping],
)

cifar_improved_test_loss, cifar_improved_test_accuracy = model_cifar_v2.evaluate(
    x_test_cifar_norm, y_test_cifar, verbose=0
)
print(f"Improved CNN test accuracy: {cifar_improved_test_accuracy:.4f}")

### Cell 14 — Evaluate the improved CNN
The learning curves show whether the added regularization narrows the training/validation gap. The printed baseline and improved test accuracies provide a direct comparison on the same held-out test set. A higher test score and a smaller generalization gap would support the recommendation.

In [ ]:
# Plot Part 2.c learning curves and compare against the baseline
plot_learning_curves(
    history_v2,
    len(history_v2.history["loss"]),
    "CIFAR-10 Improved CNN: Loss",
    "CIFAR-10 Improved CNN: Accuracy",
)

print(f"Baseline CNN test accuracy : {cifar_baseline_test_accuracy:.4f}")
print(f"Improved CNN test accuracy: {cifar_improved_test_accuracy:.4f}")

### Cell 15 — Part 2.d: Build and train VGG16 transfer learning
VGG16 is initialized with ImageNet weights and its convolutional base is frozen. CIFAR-10 images are resized from 32 × 32 to 96 × 96, then passed through VGG16's own preprocessing function. A new pooling and dense classification head is trained for the ten CIFAR-10 classes. The validation split comes from the training data; test accuracy is measured only after fitting.

In [ ]:
# Part 2.d: Transfer learning with two ImageNet-pretrained CNNs
# The CIFAR-10 source images are kept in the original 0..255 scale here.
# Each application model receives its own required preprocessing function.
# Validation is taken from the training data; the test set is used only at the end.

X_train_final = x_train_cifar.astype("float32")
X_test_final = x_test_cifar.astype("float32")
y_train_final = y_train_cifar
y_test_final = y_test_cifar

BATCH_SIZE = 128
TRANSFER_EPOCHS = 10

def build_transfer_model(model_name):
    if model_name == "VGG16":
        base_model = tf.keras.applications.VGG16(
            weights="imagenet",
            include_top=False,
            input_shape=(96, 96, 3),
        )
        preprocess_layer = tf.keras.layers.Lambda(
            tf.keras.applications.vgg16.preprocess_input,
            name="vgg16_preprocessing",
        )
    elif model_name == "MobileNetV2":
        base_model = tf.keras.applications.MobileNetV2(
            weights="imagenet",
            include_top=False,
            input_shape=(96, 96, 3),
        )
        preprocess_layer = tf.keras.layers.Lambda(
            tf.keras.applications.mobilenet_v2.preprocess_input,
            name="mobilenetv2_preprocessing",
        )
    else:
        raise ValueError("model_name must be 'VGG16' or 'MobileNetV2'")

    # Freeze the ImageNet feature extractor; train only the new CIFAR-10 head.
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Resizing(96, 96)(inputs)
    x = preprocess_layer(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name=f"{model_name}_CIFAR10")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("--- Training VGG16 transfer-learning model ---")
vgg_model = build_transfer_model("VGG16")
vgg_history = vgg_model.fit(
    X_train_final,
    y_train_final,
    epochs=TRANSFER_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=True,
)
vgg_eval = vgg_model.evaluate(X_test_final, y_test_final, verbose=0)
print(f"VGG16 test accuracy: {vgg_eval[1]:.4f}")

### Cell 16 — Train MobileNetV2 transfer learning
MobileNetV2 is a second ImageNet-pretrained feature extractor. It uses its own preprocessing function, which is different from VGG16's preprocessing. The same training/validation/test protocol and classification head are used to make the comparison more meaningful.

In [ ]:
# Train MobileNetV2 using the same train/validation/test protocol
print("--- Training MobileNetV2 transfer-learning model ---")
mobilenet_model = build_transfer_model("MobileNetV2")
mobilenet_history = mobilenet_model.fit(
    X_train_final,
    y_train_final,
    epochs=TRANSFER_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=True,
)
mobilenet_eval = mobilenet_model.evaluate(X_test_final, y_test_final, verbose=0)
print(f"MobileNetV2 test accuracy: {mobilenet_eval[1]:.4f}")

### Cell 17 — Compare pretrained models (Part 2.d)
The printed test accuracies answer the assignment's test-accuracy question. The four plots show training and validation loss/accuracy for both models. Compare final test accuracy, convergence speed, and the gap between training and validation curves. The higher test accuracy is the stronger result on this run; also consider model size and training/inference cost when discussing trade-offs.

In [ ]:
# Compare test accuracy and learning curves for both pretrained models
print("\n================ TEST ACCURACY COMPARISON ================")
print(f"VGG16       : {vgg_eval[1] * 100:.2f}%")
print(f"MobileNetV2 : {mobilenet_eval[1] * 100:.2f}%")
print("==========================================================")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
vgg_epochs = range(1, len(vgg_history.history["loss"]) + 1)
mobile_epochs = range(1, len(mobilenet_history.history["loss"]) + 1)

axes[0, 0].plot(vgg_epochs, vgg_history.history["loss"], label="Train")
axes[0, 0].plot(vgg_epochs, vgg_history.history["val_loss"], label="Validation")
axes[0, 0].set_title("VGG16 Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(vgg_epochs, vgg_history.history["accuracy"], label="Train")
axes[0, 1].plot(vgg_epochs, vgg_history.history["val_accuracy"], label="Validation")
axes[0, 1].set_title("VGG16 Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(mobile_epochs, mobilenet_history.history["loss"], label="Train")
axes[1, 0].plot(mobile_epochs, mobilenet_history.history["val_loss"], label="Validation")
axes[1, 0].set_title("MobileNetV2 Loss")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(mobile_epochs, mobilenet_history.history["accuracy"], label="Train")
axes[1, 1].plot(mobile_epochs, mobilenet_history.history["val_accuracy"], label="Validation")
axes[1, 1].set_title("MobileNetV2 Accuracy")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Written observations to complete after running the notebook

### Part 1.b — MNIST learning curve
Record the final training accuracy, validation accuracy, and test accuracy. Explain whether training and validation loss decrease together. If training accuracy is higher than validation accuracy, comment on the size of the gap and whether it suggests overfitting.

### Part 2.b — CIFAR-10 baseline learning curve
Describe whether the model continues improving on validation data throughout the 20 epochs. If training accuracy increases while validation accuracy plateaus, or validation loss begins increasing, identify this as overfitting / limited generalization.

### Part 2.c — Recommended technique
The implemented recommendation combines Batch Normalization, Dropout, L2 regularization, and Early Stopping. Compare the baseline and improved models using their held-out test accuracies and validation curves. State whether the changes helped in this run; do not claim improvement unless the measured results support it.

### Part 2.d — Pretrained-model comparison
Report the VGG16 and MobileNetV2 test accuracies from the final comparison cell. Identify which performed better on this run and compare their learning curves. Note that both use frozen ImageNet feature extractors and newly trained CIFAR-10 classification heads.

## Final submission checklist
- [ ] MNIST sample visualization included.
- [ ] MNIST CNN has four convolution layers (32, 32, 64, 64) and one 512-unit hidden dense layer.
- [ ] MNIST learning curves and written interpretation completed.
- [ ] CIFAR-10 first-nine image visualization included.
- [ ] CIFAR-10 training augmentation uses shear 0.2, zoom 0.2, and horizontal flip.
- [ ] CIFAR-10 baseline CNN has four convolution layers and dense layers of 512 and 256 units.
- [ ] SGD configuration is 0.005 learning rate, 1e-6 decay, and 0.9 momentum.
- [ ] Baseline learning curves and interpretation completed.
- [ ] Part 2.c improvement is trained and compared against the baseline.
- [ ] At least two pretrained models are trained, their learning curves are plotted, and their test accuracies are compared.
- [ ] All cells have been run successfully and the saved notebook contains fresh outputs.